In [3]:
!uv pip install transformers huggingface_hub torch accelerate

Audited 4 packages in 27ms


In [4]:
import torch

BASE_MODEL = "Qwen/Qwen3.5-2B"
 
SAE_REPO, K = "Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_100", 100
 
LAYER = 20 # which transformer layer's residual stream to read (0-63)
PROMPT = "The capital of France is"
TOP_N = 20 # how many of the active features to print

class TopKSAE:
    """Thin wrapper around the four Qwen-Scope tensors."""
 
    def __init__(self, path, k, device, dtype=torch.float32):
        sd = torch.load(path, map_location="cpu", weights_only=True)
        # compute in fp32 for numerical stability regardless of model dtype
        self.W_enc = sd["W_enc"].to(device, dtype)   # (d_sae, d_model)
        self.b_enc = sd["b_enc"].to(device, dtype)   # (d_sae,)
        self.W_dec = sd["W_dec"].to(device, dtype)   # (d_model, d_sae)
        self.b_dec = sd["b_dec"].to(device, dtype)   # (d_model,)
        self.k = k
        self.d_sae = self.W_enc.shape[0]
 
    def encode(self, residual):
        """residual (..., d_model) -> sparse feature acts (..., d_sae)."""
        x = residual.to(self.W_enc.dtype)
        pre = x @ self.W_enc.T + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        acts = torch.zeros_like(pre)
        acts.scatter_(-1, idx, vals)
        return acts
 
    def decode(self, acts):
        """sparse feature acts -> reconstructed residual (for later steering work)."""
        return acts @ self.W_dec.T + self.b_dec


In [ ]:
from huggingface_hub import login

login()


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model_kwargs = dict(dtype=torch.bfloat16, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model.eval()

sae_path = hf_hub_download(repo_id=SAE_REPO, filename=f"layer{LAYER}.sae.pt")
sae = TopKSAE(sae_path, k=K, device=device)
print(f"Loaded SAE: {SAE_REPO}  layer {LAYER}  K={K}  d_sae={sae.d_sae}")

captured = {}

def hook(_module, _inp, out):
    captured["resid"] = (out[0] if isinstance(out, tuple) else out).detach()

handle = model.model.layers[LAYER].register_forward_hook(hook)

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs)

next_id = outputs.logits[0, -1].argmax().item()
print("Next token prediction:", tokenizer.convert_ids_to_tokens([next_id])[0],
repr(tokenizer.decode([next_id])))

handle.remove()

resid = captured["resid"] # (1, seq_len, d_model)
feats = sae.encode(resid) # (1, seq_len, d_sae)

token_strs = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

last = feats[0, -1]
idx = last.nonzero(as_tuple=True)[0]
order = last[idx].argsort(descending=True)
idx = idx[order]
print(f"\nPrompt: {PROMPT!r}")
print(f"Final token: {token_strs[-1]!r}  ({len(idx)} active features)")
print(f"Top {min(TOP_N, len(idx))} features on the final token:")

for f in idx[:TOP_N]:
    print(f"  feature {int(f):>6}   act {last[f].item():.3f}")

pooled = feats[0].amax(dim=0)
top_vals, top_idx = pooled.topk(TOP_N)
print(f"\nTop {TOP_N} features across the whole prompt (max over tokens):")
for v, f in zip(top_vals.tolist(), top_idx.tolist()):
    # find which token drove this feature
    pos = feats[0, :, f].argmax().item()
    print(f"  feature {f:>6}   act {v:.3f}   peaks on {token_strs[pos]!r}")


/Users/pouyaamiri/Repos/ml-research/qwen-sparse-autoencoders/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[ERROR] `loss` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /Users/pouyaamiri/Repos/ml-research/qwen-sparse-autoencoders/.venv/lib/python3.13/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.
[ERROR] `logits` is part of Qwen3_5CausalLMOutputWithPast.__init__'s signature, but not documented. Make sure to add it to the docstring of the function in /Users/pouyaamiri/Repos/ml-research/qwen-sparse-autoencoders/.venv/lib/python3.13/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py.


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:04<00:00, 79.42it/s] 


Loaded SAE: Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_100  layer 20  K=100  d_sae=32768
Next token prediction: ĠParis ' Paris'

Prompt: 'The capital of France is'
Final token: 'Ġis'  (100 active features)
Top 20 features on the final token:
  feature    287   act 7.899
  feature   9164   act 4.729
  feature  28010   act 3.863
  feature  13601   act 3.792
  feature  15054   act 3.085
  feature  28223   act 3.038
  feature  17186   act 2.686
  feature   1538   act 2.578
  feature  25687   act 2.053
  feature  15776   act 2.020
  feature  28158   act 1.954
  feature   8070   act 1.575
  feature  18604   act 1.563
  feature  26148   act 1.439
  feature  18804   act 1.431
  feature  31073   act 1.426
  feature  11917   act 1.241
  feature   5791   act 1.200
  feature  28518   act 1.147
  feature  15743   act 1.119

Top 20 features across the whole prompt (max over tokens):
  feature   7455   act 10.759   peaks on 'Ġcapital'
  feature    287   act 7.899   peaks on 'Ġis'
  feature  18804   act 5.2

## Anger Feature Steering

The goal is now to find SAE features that distinguish angry text from calm or neutral text, then add one of those feature directions during generation to bias the model toward an angry tone.

In [6]:
# Synthetic prompts generated by gpt-5.5

angry_prompts = [
    "I am furious that you ignored every warning and made the same mistake again.",
    "This is completely unacceptable, and I am tired of pretending otherwise.",
    "I cannot believe how careless and disrespectful this whole situation has been.",
    "You wasted my time, broke your promise, and now you expect me to stay calm.",
    "The delay is outrageous, the excuses are insulting, and I want this fixed now.",
    "I am angry because nobody listened, nobody helped, and nobody took responsibility.",
    "Stop giving me vague answers and deal with the problem you created.",
    "This response is infuriating because it avoids the obvious issue.",
    "I have had enough of the incompetence and the endless excuses.",
    "The speaker is enraged, impatient, and openly frustrated with the situation.",
    "An angry customer demanded an explanation for the repeated failures.",
    "The message should sound irritated, blunt, and fed up.",
]

control_prompts = [
    "I understand the situation and would like to discuss the next steps calmly.",
    "Thank you for the update; I appreciate the clarification and your help.",
    "The meeting was moved to Thursday because several people had scheduling conflicts.",
    "The package arrived later than expected, so the customer contacted support.",
    "A neutral summary should describe the facts without emotional language.",
    "The speaker is calm, patient, and willing to resolve the issue constructively.",
    "Please explain the decision in a professional and measured tone.",
    "The report lists the causes of the delay and recommends improvements.",
    "I am disappointed, but I want to understand what happened before responding.",
    "A polite customer asked for an explanation about the shipping delay.",
    "The message should sound balanced, clear, and respectful.",
    "The answer should be concise and emotionally neutral.",
]

inspection_prompt = "I am furious that this careless mistake happened again."
steering_prompts = [
    "Respond in two sentences to this customer: My package is late again.",
    "Write a short note about a meeting being delayed.",
    "Explain why a software bug should be fixed soon.",
]


In [7]:
from dataclasses import dataclass

@dataclass
class FeatureScore:
    feature_id: int
    score: float
    positive_mean: float
    control_mean: float

class ResidualFeatureReader:
    def __init__(self, model, tokenizer, sae, layer_idx):
        self.model = model
        self.tokenizer = tokenizer
        self.sae = sae
        self.layer_idx = layer_idx
        self.layer = model.model.layers[layer_idx]

    def run(self, prompt):
        captured = {}

        def hook(_module, _inp, out):
            captured["resid"] = (out[0] if isinstance(out, tuple) else out).detach()

        handle = self.layer.register_forward_hook(hook)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        handle.remove()

        feats = self.sae.encode(captured["resid"])
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        return {
            "prompt": prompt,
            "inputs": inputs,
            "logits": outputs.logits,
            "resid": captured["resid"],
            "features": feats,
            "tokens": tokens,
        }

    def pooled_features(self, prompt, mode="max"):
        features = self.run(prompt)["features"][0]
        if mode == "max":
            return features.amax(dim=0)
        if mode == "mean":
            return features.mean(dim=0)
        if mode == "final":
            return features[-1]
        raise ValueError(f"unknown pooling mode: {mode}")

class ContrastiveFeatureFinder:
    def __init__(self, reader):
        self.reader = reader

    def score(self, positive_prompts, control_prompts, pooling="max", top_n=20):
        pos = torch.stack([self.reader.pooled_features(p, mode=pooling) for p in positive_prompts])
        ctrl = torch.stack([self.reader.pooled_features(p, mode=pooling) for p in control_prompts])
        scores = pos.mean(dim=0) - ctrl.mean(dim=0)
        vals, ids = scores.topk(top_n)

        rows = []
        for value, feature_id in zip(vals.tolist(), ids.tolist()):
            rows.append(FeatureScore(
                feature_id=feature_id,
                score=value,
                positive_mean=pos[:, feature_id].mean().item(),
                control_mean=ctrl[:, feature_id].mean().item(),
            ))
        return rows, pos, ctrl

    @staticmethod
    def print_scores(rows):
        for row in rows:
            print(
                f"feature {row.feature_id:>6} score {row.score:.3f} "
                f"angry {row.positive_mean:.3f} control {row.control_mean:.3f}"
            )

class FeatureInspector:
    def __init__(self, reader):
        self.reader = reader

    def print_token_activations(self, prompt, feature_ids, min_activation=0.0):
        result = self.reader.run(prompt)
        features = result["features"][0]
        print(prompt)
        for feature_id in feature_ids:
            print(f"\nfeature {feature_id}")
            any_printed = False
            for token, activation in zip(result["tokens"], features[:, feature_id]):
                value = activation.item()
                if value > min_activation:
                    print(f"  {token:>16}  {value:.3f}")
                    any_printed = True
            if not any_printed:
                print("  no activations above threshold")

class FeatureSteerer:
    def __init__(self, model, tokenizer, sae, layer_idx):
        self.model = model
        self.tokenizer = tokenizer
        self.sae = sae
        self.layer = model.model.layers[layer_idx]

    def feature_vector(self, feature_id):
        layer_device = next(self.layer.parameters()).device
        model_dtype = next(self.layer.parameters()).dtype
        return self.sae.W_dec[:, feature_id].to(layer_device, dtype=model_dtype)

    def weighted_vector(self, feature_scores, normalize=True):
        vec = None
        for row in feature_scores:
            part = row.score * self.feature_vector(row.feature_id)
            vec = part if vec is None else vec + part
        if normalize:
            vec = vec / vec.norm().clamp_min(1e-6)
        return vec

    def generate_with_vector(self, prompt, steer_vec, alpha, max_new_tokens=180, do_sample=False):
        def steering_hook(_module, _inp, out):
            hidden = out[0] if isinstance(out, tuple) else out
            hidden = hidden.clone()
            hidden[:, -1, :] += alpha * steer_vec
            return (hidden,) + out[1:] if isinstance(out, tuple) else hidden

        handle = self.layer.register_forward_hook(steering_hook)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        generation_kwargs = dict(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            use_cache=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        if do_sample:
            generation_kwargs.update(temperature=0.8, top_p=0.95)

        try:
            with torch.no_grad():
                out = self.model.generate(**generation_kwargs)
        finally:
            handle.remove()

        return self.tokenizer.decode(out[0], skip_special_tokens=True)

    def generate(self, prompt, feature_id, alpha, max_new_tokens=180, do_sample=False):
        return self.generate_with_vector(
            prompt,
            self.feature_vector(feature_id),
            alpha=alpha,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )
reader = ResidualFeatureReader(model, tokenizer, sae, LAYER)
finder = ContrastiveFeatureFinder(reader)
inspector = FeatureInspector(reader)
steerer = FeatureSteerer(model, tokenizer, sae, LAYER)


In [9]:
anger_scores, angry_acts, control_acts = finder.score(
    angry_prompts,
    control_prompts,
    pooling="max",
    top_n=20,
)

print("Top anger candidate features:")
finder.print_scores(anger_scores)

anger_feature_ids = [row.feature_id for row in anger_scores]


Top anger candidate features:
feature  10532 score 2.492 angry 2.980 control 0.488
feature   7399 score 1.972 angry 1.972 control 0.000
feature  26914 score 1.720 angry 2.103 control 0.383
feature   4473 score 1.439 angry 1.859 control 0.420
feature  29392 score 1.213 angry 2.171 control 0.958
feature  12676 score 1.188 angry 1.445 control 0.257
feature  19794 score 1.187 angry 1.790 control 0.603
feature  17378 score 1.187 angry 1.187 control 0.000
feature   8530 score 1.177 angry 1.177 control 0.000
feature  18756 score 1.163 angry 1.163 control 0.000
feature   1880 score 1.154 angry 1.194 control 0.040
feature  19575 score 1.145 angry 1.337 control 0.193
feature  24944 score 1.097 angry 1.135 control 0.039
feature  26805 score 1.064 angry 1.064 control 0.000
feature  17215 score 1.054 angry 1.403 control 0.349
feature  13458 score 1.045 angry 1.660 control 0.615
feature  28511 score 1.030 angry 1.030 control 0.000
feature  21689 score 1.027 angry 1.156 control 0.129
feature  25586 s

In [ ]:
# Inspect where the candidate features fire inside an angry sentence.
# Good candidates should activate on anger-bearing tokens or angry phrasing,
# not only on punctuation or generic sentence structure.
inspector.print_token_activations(
    inspection_prompt,
    anger_feature_ids[:8],
    min_activation=0.25,
)


# Findings

At a high enough alpha (e.g. 12 and 16) the model definitley is thinking about concepts like angry.

Specifically feature 10532 seems to be the most responsive to steering among all tested features. Increasing the emphasis on this feature by too much leads to the model obsessing over it (alpha=16).

### Examples

#### alpha = 12

PROMPT: Respond in two sentences to this customer: My package is late again.

<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Target: Respond to a customer.
    *   Content: The customer is angry/upset because their package is late again.
    *   Constraint: Respond in exactly two sentences.

2.  **Identify Key Angles to Address:**
    *   Apologize for the frustration.
    *   Acknowledge the delay.
    *   Offer a solution or next steps (e.g., tracking, refund, anger management).
    *   Maintain a professional and empathetic tone.

3.  **Drafting - Attempt 1:**
    I am very sorry to hear that your package is late again. We are working hard to get it to you as soon as possible.
    *Critique:* Too short, doesn't offer a solution, feels a bit passive.

4.  **Drafting - Attempt 2:**
    I sincerely apologize for the frustration caused by the delay in your package. We are currently investigating the issue and



> Notice the model's focus on concepts like agner management and frustration.

#### alpha = 16

PROMPT: Respond in two sentences to this customer: My package is late again.

<think>
Thinking Process:
1.  **Angry/ Frustrated Tone:** The customer is angry ("late again").
2.  **Goal:** Respond in two sentences.
3.  **Tone:** Apologizing, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry, angry.

In [10]:
# The model spends most of its token budget on thinking so we try to disable thinking to see
# impact of steering on the final response.
def generate_chat_with_vector(prompt, steer_vec, alpha, max_new_tokens=320, do_sample=False):
    def steering_hook(_module, _inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        hidden = hidden.clone()
        hidden[:, -1, :] += alpha * steer_vec
        return (hidden,) + out[1:] if isinstance(out, tuple) else hidden

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=0.8, top_p=0.95)

    handle = steerer.layer.register_forward_hook(steering_hook)
    try:
        with torch.no_grad():
            out = model.generate(**generation_kwargs)
    finally:
        handle.remove()

    generated_ids = out[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def generate_chat(prompt, feature_id, alpha, max_new_tokens=320, do_sample=False):
    return generate_chat_with_vector(
        prompt,
        steerer.feature_vector(feature_id),
        alpha=alpha,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
    )

anger_bundle = steerer.weighted_vector(anger_scores[:6])

print("Single strongest feature:", anger_feature_ids[0])
for alpha in [0, 8, 12, 16, 24, 32]:
    print("\n" + "=" * 80)
    print(f"single feature {anger_feature_ids[0]} alpha={alpha}")
    for prompt in steering_prompts:
        text = generate_chat(prompt, feature_id=anger_feature_ids[0], alpha=alpha)
        print(f"\nPROMPT: {prompt}")
        print(text)

print("\n" + "#" * 80)
print("Composite anger vector from top 6 features")
for alpha in [0, 10, 20, 30, 45, 60]:
    print("\n" + "=" * 80)
    print(f"anger bundle alpha={alpha}")
    for prompt in steering_prompts:
        text = generate_chat_with_vector(prompt, anger_bundle, alpha=alpha)
        print(f"\nPROMPT: {prompt}")
        print(text)


Single strongest feature: 10532

single feature 10532 alpha=0

PROMPT: Respond in two sentences to this customer: My package is late again.
I sincerely apologize for the delay in your package, as I understand how frustrating this is. I have immediately escalated your case to our logistics team to investigate the issue and ensure it is resolved as soon as possible.

PROMPT: Write a short note about a meeting being delayed.
**Subject:** Meeting Delay Notification

Dear Team,

Please be advised that our scheduled meeting with [Client/Partner Name] has been postponed. Due to [brief reason, e.g., a technical issue / a last-minute change in schedule / a conflict with another commitment], we will reschedule the discussion for [new date and time].

We apologize for the inconvenience and appreciate your understanding. Please let us know if you have any urgent matters to discuss prior to the new time slot.

Best regards,

[Your Name]
[Your Title]

PROMPT: Explain why a software bug should be fix

See output.txt for the full result. 

Emphasising feature 10532 at alpha=12 shows the model begins to inject "angry", "frustration" and similar vocab into its response. For example, when asked to explain why fixing a bug is important it mentions "user frustration" and "Anger and Complaints".

At higher alphas the model just starts to spit out gibberish as expected.


Right now, the steering merely shifts the model's next-token distribution to favour anger-related words. It does not shift the tone.